# Synthetic user profile analysis

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from cassandrautils import getFakerDF

sns.set_theme(style="whitegrid", context="notebook")
plt.rcParams["figure.figsize"] = (12, 5)
pd.set_option("display.max_columns", 30)

## 1. Load and overview

In [ ]:
users = getFakerDF()
assert users is not None and not users.empty, "fakerdata is empty - is the faker producer + sink running?"

users["birthdate"] = pd.to_datetime(users["birthdate"].astype(str), errors="coerce")
users["registered_at"] = pd.to_datetime(users["registered_at"], errors="coerce")
today = pd.Timestamp.today().normalize()
users["age"] = ((today - users["birthdate"]).dt.days / 365.25).astype(int)

print("Users: {:,}   Columns: {}".format(len(users), users.shape[1]))
users.dtypes.rename("dtype").to_frame().T

## 2. Data quality checks

### 2.1 Completeness and uniqueness

In [ ]:
quality = pd.DataFrame({
    "non_null_pct": users.notna().mean() * 100,
    "unique_values": users.nunique(),
}).round(1)
quality["unique_pct"] = (quality["unique_values"] / len(users) * 100).round(1)
quality.sort_values("unique_pct")

In [ ]:
checks = {
    "Duplicate user_id": users["user_id"].duplicated().sum(),
    "Duplicate username": users["username"].duplicated().sum(),
    "Duplicate email": users["email"].duplicated().sum(),
    "Duplicate passport_number": users["passport_number"].duplicated().sum(),
    "Duplicate credit_card_number": users["credit_card_number"].duplicated().sum(),
}
pd.Series(checks, name="rows").to_frame()

### 2.2 Logical consistency between fields

Each field is generated **independently**, so we can catch records that look valid on their own
but are impossible when fields are compared. This kind of check is what a real pipeline would run before trusting the data:

* registered **before the user was born**
* registered **while under 13** (the minimum age on most platforms)
* the `city` and `postcode` columns do not match the city and postcode inside `address`

In [ ]:
age_at_reg = (users["registered_at"].dt.tz_localize(None) - users["birthdate"]).dt.days / 365.25

consistency = pd.Series({
    "Registered before birth": (age_at_reg < 0).sum(),
    "Registered under age 13": ((age_at_reg >= 0) & (age_at_reg < 13)).sum(),
    "City not found in address": (~users.apply(lambda r: str(r["city"]) in str(r["address"]), axis=1)).sum(),
    "Postcode not found in address": (~users.apply(lambda r: str(r["postcode"]) in str(r["address"]), axis=1)).sum(),
}, name="rows")
result = consistency.to_frame()
result["pct"] = (result["rows"] / len(users) * 100).round(1)

ax = result["pct"].sort_values().plot(kind="barh", figsize=(10, 4), color="#d1495b")
ax.set_xlabel("% of records")
ax.set_title("Logical consistency violations")
plt.show()
result

## 3. Privacy: masking personal data

The table contains **sensitive personal data** (email, phone, passport, card number, IP address).
Even for synthetic data, a dashboard or presentation should only show masked values. Below is a masked view
that is safe to display. We use it for every table shown later in this notebook.

In [ ]:
def mask_email(e):
    user, _, domain = str(e).partition("@")
    return user[:2] + "***@" + domain

def mask_tail(value, keep=4):
    s = str(value)
    return "*" * max(len(s) - keep, 0) + s[-keep:]

safe = users.assign(
    email=users["email"].map(mask_email),
    phone=users["phone"].map(lambda p: mask_tail(p, 3)),
    passport_number=users["passport_number"].map(lambda p: mask_tail(p, 2)),
    credit_card_number=users["credit_card_number"].map(mask_tail),
    ip_address=users["ip_address"].astype(str).str.replace(r"\.\d+\.\d+$", ".x.x", regex=True),
)
safe[["username", "email", "phone", "passport_number", "credit_card_provider", "credit_card_number", "ip_address"]].head(8)

## 4. Age distribution

In [ ]:
bins = [18, 25, 35, 45, 55, 65, 81]
labels = ["18-24", "25-34", "35-44", "45-54", "55-64", "65-80"]
users["age_group"] = pd.cut(users["age"], bins=bins, labels=labels, right=False)

fig, axes = plt.subplots(1, 2, figsize=(15, 5))
sns.histplot(users["age"], binwidth=2, ax=axes[0], color="#00798c")
axes[0].axvline(users["age"].median(), color="#d1495b", linestyle="--", label="median = {:.0f}".format(users["age"].median()))
axes[0].set_title("Age distribution")
axes[0].set_xlabel("Age (years)")
axes[0].legend()

users["age_group"].value_counts(sort=False).plot(kind="bar", ax=axes[1], color="#30638e")
axes[1].set_title("Users by age group")
axes[1].set_xlabel("")
axes[1].tick_params(axis="x", rotation=0)
plt.tight_layout()
plt.show()

## 5. Registration trend

In [ ]:
by_year = users["registered_at"].dt.year.value_counts().sort_index()

fig, ax = plt.subplots(figsize=(14, 5))
ax.bar(by_year.index, by_year.values, color="#8ecae6")
ax.plot(by_year.index, by_year.rolling(5, center=True, min_periods=1).mean(), color="#d1495b", linewidth=2, label="5-year rolling mean")
ax.set_title("User registrations per year")
ax.set_xlabel("Year")
ax.set_ylabel("Registrations")
ax.legend()
plt.show()

## 6. Payment cards

The share of each card provider, and how many cards on file are **already expired** or **expire within the next 12 months**.
A real business would use this to remind customers to update their payment details.

In [ ]:
users["card_expiry"] = pd.to_datetime(users["credit_card_expire"], format="%m/%y", errors="coerce") + pd.offsets.MonthEnd(0)
months_left = (users["card_expiry"] - today).dt.days / 30.44
users["card_status"] = pd.cut(months_left, bins=[-1e9, 0, 12, 1e9], labels=["Expired", "Expires < 12 months", "Valid > 12 months"])

fig, axes = plt.subplots(1, 2, figsize=(16, 5.5))
users["credit_card_provider"].value_counts().sort_values().plot(kind="barh", ax=axes[0], color="#00798c")
axes[0].set_title("Card provider share")
axes[0].set_xlabel("Users")
axes[0].set_ylabel("")

status = users["card_status"].value_counts().reindex(["Expired", "Expires < 12 months", "Valid > 12 months"])
axes[1].pie(status, labels=status.index, autopct="%1.1f%%", colors=["#d1495b", "#edae49", "#00798c"], startangle=90,
            wedgeprops={"width": 0.45})
axes[1].set_title("Card expiry status")
plt.tight_layout()
plt.show()

pd.crosstab(users["credit_card_provider"], users["card_status"], normalize="index").mul(100).round(1)

## 7. Jobs and countries

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
users["job"].value_counts().head(12).sort_values().plot(kind="barh", ax=axes[0], color="#30638e")
axes[0].set_title("Most common jobs")
axes[0].set_ylabel("")
users["country"].value_counts().head(12).sort_values().plot(kind="barh", ax=axes[1], color="#edae49")
axes[1].set_title("Most common countries")
axes[1].set_ylabel("")
plt.tight_layout()
plt.show()

print("Distinct jobs: {}, distinct countries: {}".format(users["job"].nunique(), users["country"].nunique()))